In [ ]:
!pip install rtdl_revisiting_models torch scikit-learn pandas

In [ ]:
# import libraries
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import gc
import joblib
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight
from rtdl_revisiting_models import FTTransformer

# create a class for early stopping to prevent overfitting
class EarlyStopping:
    # initializer
    def __init__(self, patience=7, path='checkpoint.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    # allow a class instance to be called as a function
    def __call__(self, val_loss, model):
        # set the best loss value
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss:
            self.counter += 1
            # stop the training when the counter reaches the set number of epochs
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
    # save the model checkpoint
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

# function to load the data
def load_data(csv_path, pkl_path):
    # read the csv file
    df = pd.read_csv(csv_path)
    # read the MFCC features
    with open(pkl_path, 'rb') as f:
        mfcc_data = pickle.load(f)
    # create a dictionary with the file name as the key and it's vector as the value
    mfcc_map = dict(zip(mfcc_data['filename'], mfcc_data['mfcc_features']))
    # filter the dataframe to ensure the IDs match
    df = df[df['Video_ID'].astype(str).isin(mfcc_map.keys())].copy()
    # define the features
    feat_cols = ['similarity', 'WER', 'correct_words_#', 'mfcc_prediction',
                 'mfcc_probability', 'mfcc_svm_decision_score']
    feats = {col: df[col].values.astype(np.float32).reshape(-1, 1) for col in feat_cols}
    # define MFCC features
    mfcc_list = [mfcc_map[str(vid)] for vid in df['Video_ID']]
    X_mfcc = np.array(mfcc_list, dtype=np.float32)
    # encode labels
    le = LabelEncoder()
    y = le.fit_transform(df['label'])
    return feats, X_mfcc, y, le

# train and evaluate the model
def train_and_evaluate(X_train, y_train, X_val, y_val, device, epochs=100, patience=7):
    # clear memory
    gc.collect()
    torch.cuda.empty_cache()

    # scale the data to have mean of 0 and standard deviation of 1
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    # get class weights to handle imbalanced data
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # create a FTTransformer model
    model = FTTransformer(
        n_cont_features=X_train_s.shape[1],
        cat_cardinalities=[],
        d_out=len(np.unique(y_train)),
        n_blocks=3,
        d_block=128,
        attention_n_heads=8,
        attention_dropout=0.2,
        ffn_d_hidden_multiplier=4/3,
        ffn_dropout=0.1,
        residual_dropout=0.0
    ).to(device)
    # create data loaders for the train and validation sets
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train)),
                              batch_size=64, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_s), torch.tensor(y_val)),
                            batch_size=64)
    # define optimizer and criterion
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    # adding a Scheduler for smoother convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    early_stopper = EarlyStopping(patience=patience, path='temp_best.pt')

    # run the training calculate loss
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            outputs = model(bx, None)
            loss = criterion(outputs, by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        scheduler.step()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for vx, vy in val_loader:
                vx, vy = vx.to(device), vy.to(device)
                v_loss += criterion(model(vx, None), vy).item()

        avg_v_loss = v_loss / len(val_loader)
        early_stopper(avg_v_loss, model)
        if early_stopper.early_stop:
            model.load_state_dict(torch.load('temp_best.pt'))
            break

        if (epoch + 1) % 10 == 0 or (epoch + 1) == epochs:
            print(f"Epoch {epoch+1}/{epochs} - train Loss: {avg_train_loss:.4f} | val loss: {avg_v_loss:.4f}")

    # evaluate the model
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_val_s).to(device), None)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[:, 1] # probabilities for the positive class

        # find the best threshold using Precision-Recall curve
        precision, recall, thresholds = precision_recall_curve(y_val, probs)
        # handle division by 0
        f1_scores = np.divide(2 * precision * recall, precision + recall,
                              out=np.zeros_like(precision), where=(precision + recall) != 0)
        accs = [(probs >= t).astype(int) for t in thresholds]
        acc_scores = [accuracy_score(y_val, p) for p in accs]
        best_threshold = thresholds[np.argmax(acc_scores)]

        # apply the best threshold
        preds = (probs >= best_threshold).astype(int)

    # define the metrics
    metrics = {
        "Accuracy": accuracy_score(y_val, preds),
        "Precision": precision_score(y_val, preds, average='weighted'),
        "Recall": recall_score(y_val, preds, average='weighted'),
        "F1": f1_score(y_val, preds, average='weighted'),
        "ROC-AUC": roc_auc_score(y_val, probs),
        "Best_Threshold": best_threshold,
        "ConfMatrix": confusion_matrix(y_val, preds)
    }
    # return results
    return model, scaler, metrics

# define the device and load the data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feats_dict, X_mfcc, y, le = load_data('master.csv', 'mfcc_data.pkl')

# split the data into 60% training, 20% validation, 20% testing
idx_full = np.arange(len(y))
idx_train_val, idx_test = train_test_split(idx_full, test_size=0.20, random_state=42, stratify=y)
idx_train, idx_val = train_test_split(idx_train_val, test_size=0.25, random_state=42, stratify=y[idx_train_val])

# test configurations for combinations of features
test_configs = [
    ("1. MFCC + WER + Sim", lambda idx: np.hstack([X_mfcc[idx], feats_dict['WER'][idx], feats_dict['similarity'][idx]])),
    ("2. MFCC + WER + Dec", lambda idx: np.hstack([X_mfcc[idx], feats_dict['WER'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("3. WER + Sim + Dec", lambda idx: np.hstack([feats_dict['WER'][idx], feats_dict['similarity'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("4. Pred + Prob + Dec", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], feats_dict['mfcc_probability'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("5. All Features", lambda idx: np.hstack([X_mfcc[idx]] + [v[idx] for v in feats_dict.values()])),
    ("6. WER + Sim", lambda idx: np.hstack([feats_dict['WER'][idx], feats_dict['similarity'][idx]])),
    ("7. MFCC + WER", lambda idx: np.hstack([X_mfcc[idx], feats_dict['WER'][idx]])),
    ("8. MFCC + Sim", lambda idx: np.hstack([X_mfcc[idx], feats_dict['WER'][idx]])),
    ("9. MFCC + Dec", lambda idx: np.hstack([X_mfcc[idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("10. WER + Dec", lambda idx: np.hstack([feats_dict['WER'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("11. Sim + Dec", lambda idx: np.hstack([feats_dict['similarity'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("12. Pred + Prob", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], feats_dict['mfcc_probability'][idx]])),
    ("13. Pred + Dec", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("14. Prob + Dec", lambda idx: np.hstack([feats_dict['mfcc_probability'][idx], feats_dict['mfcc_svm_decision_score'][idx]])),
    ("15. Pred + MFCC", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], X_mfcc[idx]])),
    ("16. Prob + MFCC", lambda idx: np.hstack([feats_dict['mfcc_probability'][idx], X_mfcc[idx]])),
    ("17. Dec + MFCC ", lambda idx: np.hstack([feats_dict['mfcc_svm_decision_score'][idx], X_mfcc[idx]])),
    ("18. Pred + WER", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], feats_dict['WER'][idx]])),
    ("19. Pred + Sim", lambda idx: np.hstack([feats_dict['mfcc_prediction'][idx], feats_dict['similarity'][idx]])),
]

# evaluate the performance on each selected feature combination and select the best
selection_results = []
for name, get_data in test_configs:
    print(f"Testing: {name}")
    _, _, met = train_and_evaluate(get_data(idx_train), y[idx_train], get_data(idx_val), y[idx_val], device)
    met["Combination"] = name
    met["func"] = get_data
    selection_results.append(met)

# display Table
res_df = pd.DataFrame(selection_results).drop(columns=['ConfMatrix', 'func'])
print("\n--- PHASE 1: SELECTION METRICS ---\n", res_df.to_string(index=False))

# pick best based on F1
best_run = max(selection_results, key=lambda x: x['F1'])
print(f"\nWinner: {best_run['Combination']}")


In [ ]:
# final test using the best selected featurs
print(f"\nFinal training on {best_run['Combination']} with Optimal Threshold...")
f_model, f_scaler, f_metrics = train_and_evaluate(best_run['func'](idx_train_val), y[idx_train_val],
                                                  best_run['func'](idx_test), y[idx_test], device, epochs=100)

# print best threshold for prediction and the metrics
print(f"\nFinal Results using Threshold: {f_metrics['Best_Threshold']:.4f}")
print("\n" + "="*30)
print("FINAL HELD-OUT TEST RESULTS")
print("="*30)
for k, v in f_metrics.items():
    if k != "ConfMatrix": print(f"{k}: {v:.4f}")
print("\nConfusion Matrix:\n", f_metrics["ConfMatrix"])

# save threshold metadata along with artifacts
joblib.dump(f_metrics['Best_Threshold'], 'best_threshold.pkl')
torch.save(f_model.state_dict(), 'best_model.pt')
joblib.dump(f_scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

# Inference

In [ ]:
# --- SETUP ---
# 1. Load the tools
scaler = joblib.load('scaler.pkl')
le = joblib.load('label_encoder.pkl')
threshold = joblib.load('best_threshold.pkl')

# 2. Re-initialize the model architecture (Must match training exactly!)
model = FTTransformer(
        n_cont_features=52,             # MFCC(50) + WER(1) + Sim(1) = 52
        cat_cardinalities=[],
        d_out=2,                        # 2 classes (0/1)
        n_blocks=3,
        d_block=128,
        attention_n_heads=8,
        attention_dropout=0.2,
        ffn_d_hidden_multiplier=4/3,
        ffn_dropout=0.1,
        residual_dropout=0.0
    ).to(device)
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# --- PREDICTION ---
# 3. Scale the new data
X_new_scaled = scaler.transform(X_new_raw)

# 4. Get Probability
with torch.no_grad():
    logits = model(torch.tensor(X_new_scaled).to(device), None)
    prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

# 5. Apply the Optimal Threshold
final_prediction = (prob >= threshold).astype(int)
label = le.inverse_transform(final_prediction) # to give labels instead of numbers